<a href="https://colab.research.google.com/github/yangziyi81/yangziyi81.github.io/blob/master/logit_nested_logit_pyblp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyblp

In [ ]:
import pyblp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

In [ ]:
product_data = pd.read_csv(pyblp.data.NEVO_PRODUCTS_LOCATION)
product_data.to_csv('temp.csv')

In [ ]:
product_data

In [ ]:
# List the unique firm IDs in the product data
unique_firms = product_data['firm_ids'].unique()
print("Unique firm IDs:", sorted(unique_firms))

In [ ]:
logit_formulation = pyblp.Formulation('prices', absorb='C(product_ids)')
logit_formulation

In [ ]:
problem = pyblp.Problem(logit_formulation, product_data)
problem

In [ ]:
logit_results = problem.solve()
logit_results

#Elasticities and Diversion Ratios

In [ ]:
elasticities = logit_results.compute_elasticities()
diversions = logit_results.compute_diversion_ratios()

In [ ]:
single_market = product_data['market_ids'] == 'C61Q1'
plt.colorbar(plt.matshow(elasticities[single_market]))

In [ ]:
plt.colorbar(plt.matshow(diversions[single_market]))

#Another Way to Visualize the Elasticities

In [ ]:
means = logit_results.extract_diagonal_means(elasticities)

plt.hist(
    [means.flatten()],
    color=['blue'],
    bins=50,
)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(['Mean Own Elasticities'])

#Marginal Costs and Markups

In [ ]:
costs = logit_results.compute_costs()
plt.hist(costs, bins=50)
plt.legend(["Marginal Costs"])
plt.grid(True, linestyle='--', alpha=0.6)

Markups

In [ ]:
markups = logit_results.compute_markups(costs=costs)
plt.hist(markups, bins=50)
plt.legend(["Markups"])
plt.grid(True, linestyle='--', alpha=0.6)

#Mergers

In [ ]:
hhi = logit_results.compute_hhi()
profits = logit_results.compute_profits(costs=costs)
cs = logit_results.compute_consumer_surpluses()

Create a new set of firm IDs that represent a merger of firms 2 and 1.

In [ ]:
product_data['merger_ids'] = product_data['firm_ids'].replace(2, 1)

Solve for new eqm prices

In [ ]:
changed_prices = logit_results.compute_prices(
    firm_ids=product_data['merger_ids'],
    costs=costs
)

Compute new shares and new HHI

In [ ]:
changed_shares = logit_results.compute_shares(changed_prices)

changed_hhi = logit_results.compute_hhi(
    firm_ids=product_data['merger_ids'],
    shares=changed_shares
)
plt.hist(changed_hhi - hhi, bins=50)
plt.legend(["HHI Changes"])
plt.grid(True, linestyle='--', alpha=0.6)

Markups, ℳ
, and profits, 𝜋
, generally increase as well.

In [ ]:
changed_markups = logit_results.compute_markups(changed_prices, costs)
plt.hist(changed_markups - markups, bins=50)
plt.legend(["Markup Changes"])
plt.grid(True, linestyle='--', alpha=0.6)

# Nested logit

In [ ]:
def solve_nl(df):
    groups = df.groupby(['market_ids', 'nesting_ids'])
    df['demand_instruments20'] = groups['shares'].transform(np.size)
    nl_formulation = pyblp.Formulation('0 + prices')
    problem = pyblp.Problem(nl_formulation, df)
    return problem.solve(rho=0.7)

In [ ]:
df1 = product_data.copy()
df1['nesting_ids'] = 0
nl_results1 = solve_nl(df1)
nl_results1

In [ ]:
nl_results1.problem

In [ ]:
df2 = product_data.copy()
df2['nesting_ids'] = df2['mushy']
nl_results2 = solve_nl(df2)
nl_results2

In [ ]:
nl_results2.problem